# Comparación de resultados: nuestro Montecarlo frente al que nos dieron

Ejecutamos nuestro código (montecarlo.ipynb) y la función `MonteCarlo()` que nos dieron, los dos con nuestros parámetros (WMT, MRK, NVDA), y comparamos los resultados.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.linalg import cholesky

S0 = np.array([1.0, 1.0, 1.0])
mu, E, T, n_times, n_comunes, n_finales = 0.01, 3.0, 5, 20, 5, 100_000
dt = T / n_times
sigma = np.array([0.21915220410541955, 0.23593283496457446, 0.5174176492623017])
rho = np.array([[1.0, 0.23448266076590554, 0.2254536587614545],
                [0.23448266076590554, 1.0, 0.12948744681423088],
                [0.2254536587614545, 0.12948744681423088, 1.0]])

## 1. Nuestro Montecarlo (mismo código que montecarlo.ipynb)

In [2]:
L = np.linalg.cholesky(rho)
rng = np.random.default_rng(11)

Z1 = rng.standard_normal((n_comunes, 3)) @ L.T
e1 = np.exp((mu - 0.5 * sigma**2) * dt + sigma * Z1 * np.sqrt(dt))
S1 = np.cumprod(np.vstack([S0, e1]), axis=0)

Z2 = rng.standard_normal((n_finales, n_times - n_comunes, 3)) @ L.T
e2 = np.exp((mu - 0.5 * sigma**2) * dt + sigma * Z2 * np.sqrt(dt))
S2 = S1[-1] * np.cumprod(e2, axis=1)

X_nuestro = np.concatenate([np.broadcast_to(S1[1:], (n_finales, n_comunes, 3)), S2], axis=1)
payoff_nuestro = np.maximum(np.exp(np.log(X_nuestro / S0).mean(axis=1)).sum(axis=1) - E, 0)

## 2. Montecarlo que nos dan

In [3]:
def MonteCarlo(initial_maturity, S_0, num_sims, num_assets, num_asian_dates, value_date_index, correl_matrix, risk_free_rate, vols):
    init_time_array = np.linspace(0, initial_maturity, num_asian_dates)
    delta_t = initial_maturity / (num_asian_dates - 1)
    num_steps = value_date_index
    num_remaining_steps = num_asian_dates - value_date_index - 1
    inc_W = np.random.normal(size=(num_assets, num_steps), scale=np.sqrt(delta_t))
    m = cholesky(correl_matrix)
    inc_W_correl = np.matmul(m, inc_W)
    inc_W_remaining = np.random.normal(size=(num_sims, num_remaining_steps, num_assets), scale=np.sqrt(delta_t))
    inc_W_correl_remaining = np.matmul(inc_W_remaining, m.T)
    inc_W_correl_remaining = inc_W_correl_remaining.transpose([0, 2, 1])
    aux = np.repeat(inc_W_correl[None, ...], num_sims, axis=0)
    inc_W_correl_total = np.concatenate((aux, inc_W_correl_remaining), axis=2)
    gross_rets_total = np.exp((risk_free_rate - 0.5 * vols**2) * delta_t + vols * inc_W_correl_total)
    S_T = np.cumprod(np.concatenate((np.repeat(S_0.reshape(-1, 1)[None, ...], num_sims, axis=0), gross_rets_total), axis=2), axis=2)
    rets = S_T[:, :, 1:] / np.repeat(S_0.reshape(-1, 1)[None, ...], num_sims, axis=0)
    payoff = np.maximum(np.sum(np.prod(rets, axis=(2))**(1 / (num_asian_dates - 1)), axis=1) - 3, 0)   # corregido: 1/20
    premium = np.mean(payoff * np.exp(-risk_free_rate * (initial_maturity - init_time_array[value_date_index])))
    rets = rets.transpose([0, 2, 1])
    return rets, payoff, premium

np.random.seed(11)
X_dado, payoff_dado, premium_dado = MonteCarlo(T, S0, n_finales, 3, n_times + 1, n_comunes, rho, mu, sigma.reshape(-1, 1))

## 3. Resultados finales

In [4]:
# Sin comienzo común (todo aleatorio) para comparar en igualdad de condiciones
Z = np.random.default_rng(1).standard_normal((n_finales, n_times, 3)) @ L.T
X0_nuestro = np.cumprod(np.exp((mu - 0.5 * sigma**2) * dt + sigma * Z * np.sqrt(dt)), axis=1)
np.random.seed(1)
X0_dado = MonteCarlo(T, S0, n_finales, 3, n_times + 1, 0, rho, mu, sigma.reshape(-1, 1))[0]

casos = {("Con comienzo común (t=5)", "Nuestro"): (X_nuestro, 5), ("Con comienzo común (t=5)", "Dado"): (X_dado, 5),
         ("Sin comienzo común (t=0)", "Nuestro"): (X0_nuestro, 0), ("Sin comienzo común (t=0)", "Dado"): (X0_dado, 0)}
filas = {}
for caso, (X, t) in casos.items():
    payoff = np.maximum(np.exp(np.log(X).mean(axis=1)).sum(axis=1) - E, 0)
    valor = payoff * np.exp(-mu * (n_times - t) * dt)
    filas[caso] = {"prima": valor.mean(), "error estándar": valor.std() / np.sqrt(n_finales),
                   "payoff medio": payoff.mean(), "% payoff > 0": 100 * (payoff > 0).mean()}
pd.DataFrame(filas).T.round(4)

prima  error estándar  payoff medio  \
Con comienzo común (t=5) Nuestro  0.3052          0.0015        0.3168   
                         Dado     0.0111          0.0002        0.0115   
Sin comienzo común (t=0) Nuestro  0.3080          0.0020        0.3238   
                         Dado     0.3071          0.0020        0.3229   

                                  % payoff > 0  
Con comienzo común (t=5) Nuestro        53.433  
                         Dado            4.198  
Sin comienzo común (t=0) Nuestro        38.922  
                         Dado           38.762

## Conclusión

- **Con comienzo común las primas no coinciden.** Cada código sortea su propio comienzo (los 5 primeros trimestres) y la prima depende mucho de él.
- **Sin comienzo común las dos primas son iguales salvo el azar** (diferencia dentro del error estándar): los dos códigos calculan lo mismo.

**Por eso vamos a trabajar con nuestros datos:** dan el mismo resultado que el código que nos dieron (ya corregido), y además nuestro Montecarlo está documentado y es reproducible con su semilla.